# C11-neural-training — Review

Use this after all five sessions. First complete the 14-item self-quiz without
opening a lesson. Answers are collected in one collapsed section at the end.

The review consolidates the eight owned concepts—**softmax**,
**cross-entropy loss**, **manual backpropagation**, **autograd training**,
**torch optimizers**, **trained MLP**, **batch normalization**, and
**dropout**—under four audit contracts: numerical stability, shape closure,
training lifecycle, and mode/state behavior.


In [ ]:
import numpy as np
import torch

SEED = 20260804
ATOL = 1e-9
RTOL = 1e-7
np.random.default_rng(SEED)
torch.manual_seed(SEED)
torch.set_default_dtype(torch.float64)
print("C11 review environment ready")

## Concept summary

| Owned concept | What you must be able to do | Nonnegotiable audit |
|---|---|---|
| softmax | map logits $(N,C)$ to row probabilities | subtract each row maximum; rows sum to one |
| cross-entropy-loss | compute mean NLL and fused gradient | stable log-sum-exp; $(P-Y)/N$ for mean loss |
| manual-backpropagation | reverse a dependency graph and derive a two-layer MLP | reverse order, add branch contributions, match every shape |
| autograd-training | construct loss graphs and inspect leaf gradients | `backward()` accumulates into leaf `.grad` |
| torch-optimizers | execute and explain the update state machine | `zero_grad → forward → loss → backward → step` |
| trained-mlp | certify learning beyond “the code ran” | finite falling loss, parameter movement, named decision behavior, fixed seed |
| batch-normalization | normalize, derive gradients, and track affine/running state | batch statistics in train; running buffers in eval; parameters ≠ buffers |
| dropout | state and use inverted dropout | $M/q$ in train gives $\mathbb E[Y]=x$; identity in eval |

## Formula and API sheet

For logits $Z(N,C)$:

$$P_{ic}=\frac{e^{Z_{ic}-m_i}}{\sum_j e^{Z_{ij}-m_i}},
\quad m_i=\max_j Z_{ij},$$
$$L=-\frac1N\sum_i\log P_{i,y_i},\quad
J_{ck}=p_c(\delta_{ck}-p_k),\quad
\frac{\partial L}{\partial Z}=\frac{P-Y}{N}.$$

For $X(N,D)\to H\to C$:

$$Z_1=XW_1^\top+b_1, A_1=\operatorname{ReLU}(Z_1),Z_2=A_1W_2^\top+b_2,$$
$$G_2=(P-Y)/N,\quad dW_2=G_2^\top A_1,\quad db_2=\sum_iG_{2,i:},$$
$$G_1=(G_2W_2)\odot\mathbf1[Z_1>0],\quad
dW_1=G_1^\top X,\quad db_1=\sum_iG_{1,i:}.$$

BatchNorm for each feature:

$$\widehat X=(X-\mu)(v+\varepsilon)^{-1/2},\quad
Y=\gamma\widehat X+\beta,$$
$$dX=\frac{\gamma(v+\varepsilon)^{-1/2}}N
\left(NG-\sum_iG_i-\widehat X\sum_i(G_i\widehat X_i)\right).$$

Inverted dropout: $M\sim\mathrm{Bernoulli}(q)$,
$Y=M\odot X/q$ in train; $Y=X$ in eval.

Pinned PyTorch lifecycle:

```python
model.train()
optimizer.zero_grad(set_to_none=True)
logits = model(X)
loss = criterion(logits, y)
loss.backward()
optimizer.step()

model.eval()
with torch.no_grad():
    validation_logits = model(X_validation)
```

State map: trainable weights, BatchNorm $\gamma$, and $\beta$ are
**parameters**; running mean/variance and batch counter are **buffers**;
momentum/Adam moments belong to **optimizer state**.


## Common pitfalls and exam connections

| Symptom | Likely cause | Repair |
|---|---|---|
| softmax returns NaN on large logits | naive exponentials overflow | subtract row maxima |
| probabilities do not sum rowwise | wrong axis or extra batch scaling | normalize axis 1 with kept dimension |
| all parameter gradients are too small by $N$ | mean factor applied twice | divide once at $(P-Y)/N$ |
| one branch's gradient vanishes | overwrite instead of accumulate | add all downstream contributions |
| loss changes but weights do not | missing/misordered `backward` or `step` | audit the five-line lifecycle and movement |
| second step is unexpectedly huge | gradients were not cleared | `zero_grad` before new forward/backward |
| validation changes on repeat | model still in train mode | `eval()` plus `no_grad()` |
| validation mutates state | BatchNorm running buffers update in train mode | snapshot buffers and evaluate in eval mode |
| dropout evaluation is too small | multiplied by keep probability again | inverted dropout makes eval the identity |

Round 1 questions typically compress one of these into a code trace, a
normal-form derivative, a tensor-shape audit, or a scenario diagnosis.
Always state: loss reduction (sum or mean), shape of every gradient, current
mode, and which state owner may mutate.

## Forward links

- **C7-cnn-transfer:** the same loss, optimizer order, BatchNorm buffers,
  dropout mode, and evaluation audit apply to convolutional feature tensors.
- Later Round 2 sequence and transformer units reuse the same
  logits-to-loss and autograd state machine with different internal blocks.


## Self-quiz (14 items)

**Q1 — softmax.** Compute stable probabilities for
$(1000+\log 3,1000)$ without evaluating a huge exponential.

**Q2 — softmax shape.** For logits $(8,5)$, state the shapes of row maxima,
denominators, and probabilities when broadcasting is explicit.

**Q3 — cross-entropy.** Using Q1 probabilities and target class 1, give the
loss and the two logit-gradient components for a single example.

**Q4 — Jacobian.** State $J_{ck}$ and explain why every Jacobian row sums to
zero.

**Q5 — manual backprop.** For $Z_2=A_1W_2^\top+b_2$, give $dW_2$, $db_2$,
and $dA_1$ with shapes when $A_1(N,H)$ and $Z_2(N,C)$.

**Q6 — accumulation.** If $q=a^2+3a$, give $dq/da$ and name the graph rule
that prevents one term being lost.

**Q7 — ReLU.** State this course's derivative convention at zero and explain
how a ReLU mask affects $G_1$.

**Q8 — trained MLP.** Name four pieces of evidence that certify a tiny seeded
network really trained.

**Q9 — gradient check.** Write the centered finite-difference formula and one
reason not to check at a ReLU kink.

**Q10 — autograd.** After two identical `backward()` calls on newly built
graphs without clearing, what is in a leaf's `.grad`?

**Q11 — optimizer.** Put these in order: `step`, `loss`, `zero_grad`,
`backward`, `model(X)`. Which line mutates parameters?

**Q12 — optimizer state.** Where do momentum buffers live, and what else is
needed besides model weights to resume the same training trajectory?

**Q13 — BatchNorm.** Classify `weight`, `bias`, `running_mean`,
`running_var`, and `num_batches_tracked` as parameters or buffers.
Which statistics does evaluation use?

**Q14 — dropout and modes.** With drop probability $p=0.2$ and activation
$x=5$, give the two training outputs and probabilities under inverted
dropout. Then name both calls/contexts needed for deterministic evaluation.


## What to redo for each weak spot

These pointers name the manifest register only; they reveal no statement or
solution content.

- Softmax/stability: Session 1 Sections 1–2; later use C11-p01, p05, p11,
  p14, p21.
- Cross-entropy/Jacobian/fused gradient: Session 1 Sections 3–6; later use
  C11-p02, p06, p11, p14, p21.
- Manual backprop and shapes: Session 2 Sections 1–5; later use C11-p03, p07,
  p12, p15, p22.
- Gradient checking and NumPy training: Sessions 2 Section 6 and 3; later use
  C11-p07, p15, p17, p23.
- Autograd and optimizer lifecycle: Session 4; later use C11-p08, p16, p19,
  p23, p24.
- BatchNorm derivation/state: Session 5 Sections 1–3; later use C11-p09, p13,
  p18, p20, p24.
- Dropout expectation/mode: Session 5 Sections 4–7; later use C11-p04, p10,
  p18, p20, p24.


<details>
<summary><strong>Self-quiz answers — open only after attempting all 14</strong></summary>

**A1.** Subtract the first logit: shifted logits $(0,-\log3)$, exponentials
$(1,1/3)$, probabilities $(3/4,1/4)$.

**A2.** Row maxima $(8,1)$, denominators $(8,1)$, probabilities $(8,5)$.

**A3.** Loss $-\log(1/4)=\log4$; $P-Y=(3/4,-3/4)$.

**A4.** $J_{ck}=p_c(\delta_{ck}-p_k)$. Summing across $k$ gives
$p_c(1-\sum_kp_k)=0$.

**A5.** With $G_2(N,C)$:
$dW_2=G_2^\top A_1(C,H)$,
$db_2=\sum_iG_{2,i:}(C)$,
$dA_1=G_2W_2(N,H)$.

**A6.** $2a+3$; gradient contributions from every outgoing branch add.

**A7.** $\operatorname{ReLU}'(0)=0$; multiply the incoming hidden gradient
elementwise by $\mathbf1[Z_1>0]$.

**A8.** Finite falling loss, parameter movement, correct behavior on named
inputs, and reproduction under the fixed seed/draw/loop order.

**A9.**
$[L(\theta+\varepsilon)-L(\theta-\varepsilon)]/(2\varepsilon)$.
At a kink no unique derivative exists, so the centered slope need not equal
the pinned backprop convention.

**A10.** The sum, $2g$.

**A11.** `zero_grad → model(X) → loss → backward → step`; `step` mutates
parameters.

**A12.** Optimizer state; also optimizer hyperparameters/state, relevant RNG
state, and data order are needed.

**A13.** `weight`/$\gamma$ and `bias`/$\beta$ are parameters.
`running_mean`, `running_var`, and `num_batches_tracked` are buffers.
Evaluation uses running statistics.

**A14.** Keep probability $q=0.8$: output $0$ with probability $0.2$ or
$5/0.8=6.25$ with probability $0.8$. Use `model.eval()` and
`with torch.no_grad():`.

</details>
